In [ ]:
import pandas as pd
import numpy as np

Import request

## Step 1: Mapping SNOMED CT Codes to ICD-10-CM
To evaluate whether a rural triage unit can maintain financial solvency, clinical
condition records must first be translated into billable diagnostic codes. This step
uses the UMLS REST API crosswalk endpoint (accessed August 2026) to map SNOMED CT
condition codes to ICD-10-CM.

**Method note:** This uses the UMLS `/crosswalk` endpoint, which returns codes sharing
a UMLS concept identifier—not the rule-based NLM SNOMED CT to ICD-10-CM Map. NLM
states that this synonymy has not been rigorously tested for clinical use and that
results should be reviewed for relevancy. The crosswalk carries no map groups,
priorities, or age/sex conditional rules. It was chosen because it requires only an API
key rather than a manually uploaded licensed file, allowing the notebook to run from a
cold start.

Where multiple ICD-10-CM codes are returned, the most specific (longest) is selected,
since ICD-10-CM billing requires the most specific code available—`E11` is a category, while `E11.9` is billable.

Performed strictly as a sustainability demonstration on synthetic data, not as a clinical
revenue tool.

In [ ]:
import requests
from google.colab import userdata

UMLS_KEY = userdata.get('UMLS_KEY')
UTS_BASE = "https://uts-ws.nlm.nih.gov/rest"

def snomed_to_icd10(snomed_code):
    """
    Crosswalk a SNOMED CT code to ICD-10-CM via UMLS CUI synonymy.

    Returns (best_code, all_codes, status). The 'best' code is the longest
    returned, i.e. the most specific — ICD-10-CM billing requires the most
    specific code available, so a 3-character category like 'E11' is not
    billable while 'E11.9' is.

    NOTE: This is the UMLS /crosswalk endpoint, NOT the NLM SNOMED CT to
    ICD-10-CM Map. It returns codes sharing a UMLS concept identifier, without
    map rules, groups, or priorities. NLM states this synonymy has not been
    rigorously tested for clinical use. Documented limitation.
    """
    url = f"{UTS_BASE}/crosswalk/current/source/SNOMEDCT_US/{snomed_code}"
    r = requests.get(url, params={"targetSource": "ICD10CM", "apiKey": UMLS_KEY},
                     timeout=30)
    if r.status_code == 404:
        return None, [], "No ICD-10-CM crosswalk found"
    r.raise_for_status()
    results = r.json().get('result', [])
    codes = [x['ui'] for x in results if x.get('ui')]
    if not codes:
        return None, [], "Empty result"
    best = max(codes, key=len)          # most specific
    return best, codes, "OK"


for code, label in [('44054006', 'Type 2 diabetes'), ('195967001', 'Asthma')]:
    best, codes, status = snomed_to_icd10(code)
    print(f"{label:<20} SNOMED {code} -> best={best}  all={codes}  [{status}]")

Type 2 diabetes      SNOMED 44054006 -> best=E11  all=['E11']  [OK]
Asthma               SNOMED 195967001 -> best=J45.909  all=['J45', 'J45.909', 'J45.90']  [OK]


## Step 2: Validating ICD-10-CM Codes Against Official Standards

Before any mapped diagnostic codes are passed further down the reimbursement chain, they must be verified to prevent invalid values from propagating silently. This block parses the official fixed-width text files from the [CDC/NCHS ICD-10-CM 2027 Set](https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Publications/ICD10CM/2027/icd10cm-code-descriptions-2027.zip) (accessed August 2026) to confirm that every generated code actively exists in the official federal dataset. Unmapped or unrecognized codes are explicitly tracked rather than silently dropped to ensure data transparency.

In [ ]:
import requests, zipfile, io

CDC_URL = ("https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Publications/"
           "ICD10CM/2027/icd10cm-code-descriptions-2027.zip")

r = requests.get(CDC_URL, timeout=120)
r.raise_for_status()
z = zipfile.ZipFile(io.BytesIO(r.content))
print("Files in archive:", z.namelist())



Files in archive: ['icd10cm-code-descriptions-2027/', 'icd10cm-code-descriptions-2027/icd10cm-codes-2027.txt', 'icd10cm-code-descriptions-2027/icd10cm-codes-addenda-2027.txt', 'icd10cm-code-descriptions-2027/icd10cm-order-2027.txt', 'icd10cm-code-descriptions-2027/icd10cm-order-addenda-2027.txt', 'icd10cm-code-descriptions-2027/icd10cmCodesFile.pdf', 'icd10cm-code-descriptions-2027/icd10OrderFiles.pdf']


In [ ]:
import pandas as pd

path = [n for n in z.namelist() if n.endswith('icd10cm-order-2027.txt')][0]
with z.open(path) as f:
    cdc = pd.read_fwf(f, colspecs=[(0,5),(6,13),(14,15),(16,76),(77,None)],
                      header=None, names=['order','code','billable','short','long'],
                      dtype={'code':str,'billable':str}, encoding='latin-1')

cdc['code'] = cdc['code'].str.strip()
valid_icd10 = set(cdc['code'])
billable_icd10 = set(cdc.loc[cdc['billable']=='1', 'code'])
print(f"{len(cdc):,} ICD-10-CM codes | {len(billable_icd10):,} billable")

def validate_icd10(code):
    """Returns (exists, is_billable). CDC stores codes undotted."""
    if code is None:
        return False, False
    clean = str(code).replace('.', '').strip().upper()
    return clean in valid_icd10, clean in billable_icd10

for c in ['E11', 'E11.9', 'J45.909', 'J45', 'ZZZZZ']:
    ex, bl = validate_icd10(c)
    print(f"  {c:<10} exists={ex}  billable={bl}")

98,403 ICD-10-CM codes | 74,879 billable
  E11        exists=True  billable=False
  E11.9      exists=True  billable=True
  J45.909    exists=True  billable=True
  J45        exists=True  billable=False
  ZZZZZ      exists=False  billable=False


## DRG Approximation & Payment Weight Estimate

## Step 3 & 4: MDC Approximation and Weight Estimate

*Methodological limitation:* MS-DRG assignment is performed per complete hospital
stay by a grouper considering principal diagnosis, secondary complications, and
procedures—not a single point-of-care diagnosis. These inputs are unavailable
at triage.

This notebook therefore approximates at the *Major Diagnostic Category* level.
Each validated ICD-10-CM code is mapped to its MDC using the [CMS MS-DRG Definitions Manual](https://www.cms.gov/files/zip/fy2027-fr-icd10-ms-drg-definitions-manual-files-v44.zip) (V44, FY2027), and the range of DRG relative weights within
that MDC is retrieved from the [NBER DRG weight file](https://data.nber.org/drg/csv/drgweight2026FR.csv) (FY2026 Final Rule, accessed August 2026). No single DRG is assigned.

The mean weight for the patient's principal MDC is multiplied by the FY2026 IPPS
operating standardized amount (`$6,752.61` for hospitals meeting quality-reporting and meaningful EHR-use requirements; `$6,589.72` otherwise). No wage-index
adjustment is applied, so the estimate is unadjusted for local labor costs.

### Loading file to inspect Columns

In [ ]:
# Load the file to inspect columns
NBER_URL = "https://data.nber.org/drg/csv/drgweight2026FR.csv"
nber_df = pd.read_csv(NBER_URL)
print(f"Loaded {len(nber_df)} DRGs from NBER (FY2026 Final Rule)")
print(nber_df.columns.tolist())

Loaded 773 DRGs from NBER (FY2026 Final Rule)
['ms_drg', 'pa_drg', 'nprm_drg', 'mdc', 'type', 'msdrg_title', 'weights', 'los_geo', 'los_mean']


### Crosswalk Logic

In [ ]:
import requests, zipfile, io, re, glob
from collections import defaultdict

# --- Download the CMS MS-DRG Definitions Manual (V44, FY2027) ---
MSDRG_URL = "https://www.cms.gov/files/zip/fy2027-fr-icd10-ms-drg-definitions-manual-files-v44.zip"
r = requests.get(MSDRG_URL, timeout=180); r.raise_for_status()
zdrg = zipfile.ZipFile(io.BytesIO(r.content))
mdc_files = [n for n in zdrg.namelist() if 'mdcs_' in n.lower() and n.endswith('.txt')]
print(f"MDC files: {mdc_files}")

# --- Parse ICD-10 -> MDC ---
MDC_HDR = re.compile(r'^MDC (\d{2}) Assignment of Diagnosis Codes\s*$')
CODE_LN = re.compile(r'^\s{2}([A-Z][A-Z0-9]{2,7})\s{2,}(.+?)\s*$')

icd_to_mdc = {}
for name in mdc_files:
    in_block = False; cur = None
    for raw in zdrg.open(name):
        line = raw.decode('latin-1').rstrip('\r\n')
        m = MDC_HDR.match(line)
        if m:
            cur, in_block = m.group(1), True; continue
        if line.strip() and not line.startswith('  ') and in_block:
            in_block = False; continue
        m = CODE_LN.match(line)
        if m and in_block and cur:
            icd_to_mdc.setdefault(m.group(1), cur)

print(f"{len(icd_to_mdc):,} ICD-10 codes mapped to MDC")

# --- MDC -> DRG weight range (NBER, already loaded as nber_df) ---
nber_df['mdc'] = nber_df['mdc'].astype(str).str.strip().str.zfill(2)
nber_df['weights'] = pd.to_numeric(nber_df['weights'], errors='coerce')
mdc_weights = (nber_df.dropna(subset=['weights'])
                      .groupby('mdc')['weights']
                      .agg(['min','max','mean','count']).round(4))

def icd10_to_mdc_weights(icd10_code):
    """ICD-10 -> MDC -> DRG weight range for that MDC."""
    clean = str(icd10_code).replace('.', '').strip().upper()
    mdc = icd_to_mdc.get(clean)
    if mdc is None or mdc not in mdc_weights.index:
        return None
    row = mdc_weights.loc[mdc]
    return {'mdc': mdc, 'weight_min': row['min'], 'weight_max': row['max'],
            'weight_mean': row['mean'], 'n_drgs': int(row['count'])}

for c in ['E11.9', 'J45.909', 'I10']:
    print(f"  {c:<10} {icd10_to_mdc_weights(c)}")

MDC files: ['mdcs_00_07.txt', 'mdcs_08_11.txt', 'mdcs_12_21.txt', 'mdcs_22_25.txt']
22,436 ICD-10 codes mapped to MDC
  E11.9      {'mdc': '10', 'weight_min': np.float64(0.6212), 'weight_max': np.float64(3.7268), 'weight_mean': np.float64(1.7735), 'n_drgs': 26}
  J45.909    {'mdc': '04', 'weight_min': np.float64(0.6285), 'weight_max': np.float64(6.4347), 'weight_mean': np.float64(1.5272), 'n_drgs': 41}
  I10        {'mdc': '05', 'weight_min': np.float64(0.4551), 'weight_max': np.float64(11.3188), 'weight_mean': np.float64(3.2464), 'n_drgs': 101}


## CMS Base Rate & Limitation

*Limitation: Category-level crosswalk results drop out of the estimate.*

The UMLS crosswalk sometimes returns only a 3-character ICD-10 category rather
than a billable code—type 2 diabetes (SNOMED 44054006) returns `E11` instead of `E11.9`. Such codes fail two downstream checks: they are not billable per the
CDC 2027 order file, and they carry no MDC assignment in the MS-DRG Definitions
Manual, which assigns MDCs at the billable-code level only. A condition that
crosswalks to a category therefore contributes nothing to the weight estimate.

This is a consequence of using the UMLS CUI-synonymy crosswalk rather than the
rule-based NLM SNOMED CT to ICD-10-CM Map, which returns billable targets under
its map rules. Estimates are therefore *conservative*—undercounting rather
than overstating encounter value. Mapping coverage is reported below.

In [ ]:
CMS_BASE_RATE = 6752.61   # FY2026 IPPS operating standardized amount,
                          # full update (quality reporting + EHR).
                          # Non-qualifying rate: $6,589.72.
                          # No wage-index adjustment applied.
def estimate_encounter_value(snomed_codes, age, sex):
    """
    SNOMED conditions -> ICD-10-CM -> MDC -> DRG weight range.
    Sustainability demonstration on synthetic data. Not a payment prediction.
    """
    icd10, unmapped, mdcs = [], [], []

    for code in snomed_codes:
        best, all_codes, status = snomed_to_icd10(code)
        if best is None:
            unmapped.append({'snomed': code, 'reason': status}); continue
        exists, billable = validate_icd10(best)
        if not exists:
            unmapped.append({'snomed': code, 'reason': f'{best} not in CDC 2027'}); continue
        icd10.append({'code': best, 'billable': billable})
        w = icd10_to_mdc_weights(best)
        if w:
            mdcs.append({**w, 'icd10': best})

    primary = max(mdcs, key=lambda x: x['weight_mean']) if mdcs else None

    return {
        'icd10_codes':  [c['code'] for c in icd10],
        'billable_codes': [c['code'] for c in icd10 if c['billable']],
        'unmapped':     unmapped,
        'mdc':          primary['mdc'] if primary else None,
        'weight_range': (primary['weight_min'], primary['weight_max']) if primary else None,
        'weight_mean':  primary['weight_mean'] if primary else None,
        'estimate_usd': round(primary['weight_mean'] * CMS_BASE_RATE, 2) if primary else None,
        'caveats': [
            'NON-CLINICAL USE ONLY. Synthetic Synthea data.',
            'MDC-level approximation. CMS states MS-DRG assignment requires principal '
            'diagnosis, up to 24 secondary diagnoses, up to 25 procedures, and in some '
            'cases age, sex and discharge status — inputs unavailable at triage.',
            'Weight reflects clinical resource intensity, not payment. Critical Access '
            'and Rural Emergency Hospitals are cost-reimbursed, not DRG-paid.',
            'SNOMED-to-ICD-10 via UMLS CUI crosswalk, not the rule-based NLM Map.',
            'Conditions crosswalking only to a 3-character ICD-10 category '
            '(e.g. E11) contribute nothing to the estimate — such codes are '
            'neither billable nor MDC-assigned. Estimates are conservative.',
            'National standardized amount used without wage-index adjustment. '
            'Rural New Mexico wage indices are below 1.0, so unadjusted '
            'estimates overstate facility-level payment.',
        ],
    }

In [ ]:




result = estimate_encounter_value(['44054006', '195967001'], age=84, sex='f')

for k, v in result.items():
    if k == 'caveats':
        print(f"\ncaveats:")
        for c in v: print(f"  - {c}")
    else:
        print(f"{k}: {v}")

icd10_codes: ['E11', 'J45.909']
billable_codes: ['J45.909']
unmapped: []
mdc: 04
weight_range: (np.float64(0.6285), np.float64(6.4347))
weight_mean: 1.5272
estimate_usd: 10312.59

caveats:
  - NON-CLINICAL USE ONLY. Synthetic Synthea data.
  - MDC-level approximation. CMS states MS-DRG assignment requires principal diagnosis, up to 24 secondary diagnoses, up to 25 procedures, and in some cases age, sex and discharge status — inputs unavailable at triage.
  - Weight reflects clinical resource intensity, not payment. Critical Access and Rural Emergency Hospitals are cost-reimbursed, not DRG-paid.
  - SNOMED-to-ICD-10 via UMLS CUI crosswalk, not the rule-based NLM Map.
  - Conditions crosswalking only to a 3-character ICD-10 category (e.g. E11) contribute nothing to the estimate — such codes are neither billable nor MDC-assigned. Estimates are conservative.
  - National standardized amount used without wage-index adjustment. Rural New Mexico wage indices are below 1.0, so unadjusted estim

*Methodological limitation:* MS-DRG assignment is performed by a grouper that
considers the principal diagnosis, up to 24 secondary diagnoses, up to 25
procedures, and in some cases age, sex, and discharge status (CMS, MS-DRG
Classifications and Software). These inputs are unavailable at the point of triage.

This notebook therefore approximates at the *Major Diagnostic Category* level:
each validated ICD-10-CM code is mapped to its MDC using the [CMS MS-DRG Definitions Manual](https://www.cms.gov/files/zip/fy2027-fr-icd10-ms-drg-definitions-manual-files-v44.zip) (V44, FY2027), and the range of DRG relative weights within
that MDC is reported. No single DRG is assigned. MDC assignments from the V44
(FY2027) manual are applied to FY2026 weights; MDC groupings are stable across
adjacent years. Where a patient has multiple conditions, the MDC with the highest
mean weight is used as a principal-diagnosis proxy—a stated assumption, not a
clinical determination.

## Step 5: Encounter Value Estimates

To integrate this pipeline into the existing Streamlit demonstration app without
structural refactoring, all mapping, validation, and weighting logic is wrapped
in a single function: `estimate_encounter_value(snomed_codes, age, sex)`. It
processes a patient's condition codes, tracks unmapped gaps explicitly, and
returns a formatted dictionary containing the encounter value estimate alongside
mandatory non-clinical caveats.

In [ ]:
import json

def format_encounter_result(r, case_name=None):
    """Print an encounter value estimate in readable form."""
    print("=" * 62)
    title = f"ENCOUNTER VALUE ESTIMATE{f' — {case_name}' if case_name else ''}"
    print(f"  {title}")
    print("=" * 62)

    print("\n[+] Mapped & validated ICD-10-CM codes:")
    print("\n".join(f"    • {c}" for c in r['icd10_codes']) or "    (none)")

    print("\n[+] Billable codes:")
    print("\n".join(f"    • {c}" for c in r['billable_codes']) or "    (none)")

    print("\n[-] Unmapped codes:")
    if r['unmapped']:
        for u in r['unmapped']:
            print(f"    • SNOMED {u['snomed']} — {u['reason']}")
    else:
        print("    (none)")

    print("\n[#] Grouping and weight summary:")
    print(f"    • Major Diagnostic Category : {r['mdc'] or 'N/A'}")
    if r['weight_range']:
        lo, hi = r['weight_range']
        print(f"    • DRG weight range (MDC)   : {lo:.4f} – {hi:.4f}")
        print(f"    • Mean weight              : {r['weight_mean']:.4f}")
    else:
        print("    • DRG weight range (MDC)   : N/A")
    est = r['estimate_usd']
    print(f"    • Encounter value estimate : {f'${est:,.2f}' if est else 'N/A'}")

    print("\n[!] Caveats:")
    for c in r['caveats']:
        print(f"    * {c}")
    print("=" * 62)


# Demonstrate on real output, not a hand-written sample
format_encounter_result(
    estimate_encounter_value(['44054006', '195967001'], age=84, sex='f'),
    case_name="test: diabetes + asthma"
)


  ENCOUNTER VALUE ESTIMATE — test: diabetes + asthma

[+] Mapped & validated ICD-10-CM codes:
    • E11
    • J45.909

[+] Billable codes:
    • J45.909

[-] Unmapped codes:
    (none)

[#] Grouping and weight summary:
    • Major Diagnostic Category : 04
    • DRG weight range (MDC)   : 0.6285 – 6.4347
    • Mean weight              : 1.5272
    • Encounter value estimate : $10,312.59

[!] Caveats:
    * NON-CLINICAL USE ONLY. Synthetic Synthea data.
    * MDC-level approximation. CMS states MS-DRG assignment requires principal diagnosis, up to 24 secondary diagnoses, up to 25 procedures, and in some cases age, sex and discharge status — inputs unavailable at triage.
    * Weight reflects clinical resource intensity, not payment. Critical Access and Rural Emergency Hospitals are cost-reimbursed, not DRG-paid.
    * SNOMED-to-ICD-10 via UMLS CUI crosswalk, not the rule-based NLM Map.
    * Conditions crosswalking only to a 3-character ICD-10 category (e.g. E11) contribute nothing to th

In [ ]:
# Final cell — run the estimator over the cohort
# Final cell — run the estimator over the cohort
import json
import urllib.request

# cohort_snomed.json is produced by the MAIN notebook. For cold certification
# this notebook must not depend on another notebook having run first, so a
# committed copy is fetched from GitHub when no local file exists.
COHORT_SNOMED_URL = ("https://raw.githubusercontent.com/cyeef/"
                     "Capstone-Projects/main/Cloud-App/cohort_snomed.json")

try:
    with open('cohort_snomed.json') as f:
        cohort_snomed = json.load(f)
    print("cohort_snomed.json — loaded from local runtime")
except FileNotFoundError:
    try:
        with urllib.request.urlopen(COHORT_SNOMED_URL, timeout=60) as r:
            cohort_snomed = json.loads(r.read().decode())
        print("cohort_snomed.json — fetched from committed GitHub copy")
    except Exception as e:
        raise FileNotFoundError(
            "cohort_snomed.json not found locally and not fetchable from "
            f"{COHORT_SNOMED_URL} ({e}). Either run the main notebook's "
            "export cell in this runtime, or commit the file and fix the URL."
        ) from e

estimates = {case: estimate_encounter_value(v['codes'], v['age'], v['sex'])
             for case, v in cohort_snomed.items()}
# ... rest of the cell unchanged
with open('reimbursement_estimates.json', 'w') as f:
    json.dump(estimates, f, indent=2)

for case, e in estimates.items():
    n_map = len(e['icd10_codes']); n_un = len(e['unmapped'])
    val = f"${e['estimate_usd']:,.0f}" if e['estimate_usd'] else "—"
    print(f"{case:<20} mapped={n_map:>2} unmapped={n_un:>2} MDC={e['mdc'] or '—':>3} {val}")

cohort_snomed.json — fetched from committed GitHub copy
Pediatric            mapped= 2 unmapped= 1 MDC= 01 $14,232
Healthy adult        mapped= 0 unmapped= 0 MDC=  — —
Complex geriatric    mapped=13 unmapped=15 MDC= 05 $21,922
Allergy carrier      mapped= 1 unmapped= 6 MDC= 03 $9,959
High medication      mapped=15 unmapped= 8 MDC= 05 $21,922
